# 🚀 06 - Pipeline de Producción MoE (Inferencia con Batches)

**¡ACTUALIZADO CON LAS CORRECCIONES CRÍTICAS DEL DASHBOARD!**

## 1. Configuración del Entorno y Rutas

In [ ]:
!pip install -q SimpleITK nibabel torch torchvision monai einops timm
import os
import glob
import random
import shutil
import subprocess
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import cv2
from PIL import Image
import SimpleITK as sitk
import nibabel as nib
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models
from torchvision.models.video import R3D_18_Weights, r3d_18
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from monai.networks.blocks import PatchEmbed
import timm
import math
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, WeightedRandomSampler

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# --- Rutas (igual que notebook 03): ZIPs en Drive → datasets en disco local Colab ---
RAW_DIR = "/content/drive/MyDrive/PROYECTO_MOE_VISION/01_Data/Raw/"
LOCAL_DEST = "/content/datasets/"
WEIGHTS_DIR = "/content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/"
FEATURE_DIR = WEIGHTS_DIR
os.makedirs(LOCAL_DEST, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

DATASET_ROOTS = {
    "NIH":      (os.path.join(LOCAL_DEST, "NIH Chest X ray 14"), 0),
    "ISIC":     (os.path.join(LOCAL_DEST, "ISIC 2019"), 1),
    "Osteo":    (os.path.join(LOCAL_DEST, "Knee Osteoarthritis Classification"), 2),
    "LUNA16":   (os.path.join(LOCAL_DEST, "Luna16 Lung Cancer Dataset"), 3),
    "Pancreas": (os.path.join(LOCAL_DEST, "Pancreas Cancer"), 4),
}

print("Rutas (datasets tras extracción en LOCAL_DEST):")
for k, (p, d) in DATASET_ROOTS.items():
    ok = os.path.isdir(p)
    print(f"  {k} id={d} exists={ok} -> {p}")
print(f"RAW_DIR={RAW_DIR}")
print(f"LOCAL_DEST={LOCAL_DEST}")
print(f"WEIGHTS_DIR={WEIGHTS_DIR}")


## 2. Preprocesador Adaptativo

In [ ]:
class AdaptivePreprocessor:
    """Carga archivos y retorna tensores nativos (2D o 3D)."""
    def __init__(
        self,
        size_2d=(224, 224),
        size_3d=(64, 64, 64),
        hu_window_ct=(-1000, 400),
        use_clahe_nih_osteo=True,
        use_imagenet_norm_2d=True,
    ):
        self.size_2d = size_2d
        self.size_3d = size_3d
        self.hu_min, self.hu_max = hu_window_ct
        self.use_clahe_nih_osteo = use_clahe_nih_osteo
        self.use_imagenet_norm_2d = use_imagenet_norm_2d
        self.imagenet_mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
        self.imagenet_std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)

    def __call__(self, file_path):
        ext = file_path.lower()
        if ext.endswith(('.png', '.jpg', '.jpeg')): return self._process_2d(file_path)
        if ext.endswith('.mha'):                     return self._process_mha(file_path)
        if ext.endswith(('.mhd', '.nii.gz', '.nii')): return self._process_3d(file_path)
        raise ValueError(f'Formato no soportado: {file_path}')

    def _is_nih_or_osteo(self, path: str) -> tuple[bool, bool]:
        p = str(path).replace('\\', '/').lower()
        is_nih = ('nih chest x ray 14' in p) or ('/nih/' in p)
        is_osteo = ('knee osteoarthritis classification' in p) or ('/klgrade/' in p) or ('/osteo' in p)
        return is_nih, is_osteo

    def _clahe_nih_rgb(self, img_rgb_u8: np.ndarray) -> np.ndarray:
        lab = cv2.cvtColor(img_rgb_u8, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        hist = cv2.calcHist([l], [0], None, [256], [0, 256]).flatten()
        peaks = int(np.sum(hist > hist.mean()))
        tile_size = max(2, int(np.ceil(np.log(max(peaks, 2)))))
        valleys = hist[hist <= hist.mean()]
        val_mean = float(valleys.mean()) if len(valleys) > 0 else 1.0
        clip = float(np.clip(hist.max() / (val_mean + 1e-6), 1.0, 4.0))
        clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile_size, tile_size))
        l_eq = clahe.apply(l)
        lab_eq = cv2.merge((l_eq, a, b))
        arr_eq = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB)
        return cv2.GaussianBlur(arr_eq, (5, 5), sigmaX=1.0)

    def _clahe_osteo_gray_to_rgb(self, img_rgb_u8: np.ndarray) -> np.ndarray:
        g = cv2.cvtColor(img_rgb_u8, cv2.COLOR_RGB2GRAY)
        hist = cv2.calcHist([g], [0], None, [256], [0, 256]).flatten()
        tile_size = max(2, int(np.ceil(np.log(max(int(np.sum(hist > hist.mean())), 2)))))
        valleys = hist[hist <= hist.mean()]
        val_mean = float(valleys.mean()) if len(valleys) > 0 else 1.0
        clip = float(np.clip(hist.max() / (val_mean + 1e-6), 1.0, 4.0))
        clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile_size, tile_size))
        enhanced = clahe.apply(g)
        return np.repeat(enhanced[:, :, None], 3, axis=2)

    def _apply_imagenet_norm_2d(self, t: torch.Tensor) -> torch.Tensor:
        if not self.use_imagenet_norm_2d:
            return t
        mean = self.imagenet_mean.to(dtype=t.dtype, device=t.device)
        std = self.imagenet_std.to(dtype=t.dtype, device=t.device)
        return (t - mean) / std

    def _process_2d(self, path):
        is_nih, is_osteo = self._is_nih_or_osteo(path)
        img = Image.open(path).convert('RGB').resize(self.size_2d)
        arr = np.array(img, dtype=np.uint8)

        if self.use_clahe_nih_osteo:
            if is_nih:
                arr = self._clahe_nih_rgb(arr)
            elif is_osteo:
                arr = self._clahe_osteo_gray_to_rgb(arr)

        t = torch.from_numpy(arr.astype(np.float32) / 255.0).permute(2, 0, 1).contiguous()
        return self._apply_imagenet_norm_2d(t)

    def _process_mha(self, path):
        itk_img = sitk.ReadImage(path)
        size = itk_img.GetSize()
        arr = sitk.GetArrayFromImage(itk_img).astype(np.float32)
        if len(size) == 2:                       return self._array_2d_to_tensor(arr, path_hint=path)
        if len(size) == 3 and size[2] <= 1:
            arr = np.squeeze(arr)
            if arr.ndim == 2:                    return self._array_2d_to_tensor(arr, path_hint=path)
        return self._volume_array_to_tensor(arr, path)

    def _array_2d_to_tensor(self, arr, path_hint=''):
        if arr.max() > 1.5:
            arr = arr / 255.0 if arr.max() > 2.0 else (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
        arr = arr.astype(np.float32)
        t = torch.from_numpy(arr).unsqueeze(0)
        if t.shape[1:] != torch.Size(self.size_2d):
            t = F.interpolate(t.unsqueeze(0), size=self.size_2d, mode='bilinear', align_corners=False).squeeze(0)

        # Para 2D, el router trabaja en 3 canales + normalización ImageNet.
        if t.shape[0] == 1:
            t = t.repeat(3, 1, 1)
        elif t.shape[0] > 3:
            t = t[:3]
        t = t.clamp(0.0, 1.0)
        return self._apply_imagenet_norm_2d(t)

    def _volume_array_to_tensor(self, img_arr, path_hint=''):
        amin, amax = float(np.nanmin(img_arr)), float(np.nanmax(img_arr))
        pre_norm = amax <= 1.5 and amin >= -1e-2 and 'Pancreas' in str(path_hint).replace('\\', '/')
        if not pre_norm:
            img_arr = np.clip(img_arr, self.hu_min, self.hu_max)
            img_arr = (img_arr - self.hu_min) / (self.hu_max - self.hu_min + 1e-8)
        else:
            img_arr = np.clip(img_arr, 0.0, 1.0)
        t = torch.tensor(img_arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=self.size_3d, mode='trilinear', align_corners=False)
        return t.squeeze(0)  # [1, 64, 64, 64]

    def _process_3d(self, path):
        if path.endswith('.mhd'):
            itk_img = sitk.ReadImage(path)
            img_arr = sitk.GetArrayFromImage(itk_img).astype(np.float32)
        else:
            nii_img = nib.load(path)
            img_arr = np.transpose(nii_img.get_fdata().astype(np.float32), (2, 1, 0))
        return self._volume_array_to_tensor(img_arr, path_hint=path)

preprocessor = AdaptivePreprocessor()
print('AdaptivePreprocessor listo.')

# --- Experto LUNA (R3D-18): AdaptivePreprocessor da [1,64³]; el modelo usa 3ch + stats Kinetics (LUNA16_R3D18_Training) ---
KINETICS_MEAN = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
KINETICS_STD = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)


def luna_1ch_to_kinetics_3ch(t, device=None):
    """[B,1,D,H,W] en [0,1] → [B,3,D,H,W] con normalización Kinetics-400."""
    if device is None:
        device = t.device
    mean = KINETICS_MEAN.to(device=device, dtype=t.dtype)
    std = KINETICS_STD.to(device=device, dtype=t.dtype)
    t3 = t.expand(-1, 3, -1, -1, -1)
    return (t3 - mean) / std


VALID_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.mha', '.mhd', '.nii', '.nii.gz')

def scan_dataset_files(root_path, max_files=None):
    """Escanea recursivamente y retorna paths de archivos médicos válidos."""
    files = []
    for dirpath, _, filenames in os.walk(root_path):
        for fname in sorted(filenames):
            if fname.lower().endswith(VALID_EXTENSIONS):
                files.append(os.path.join(dirpath, fname))
    if max_files:
        files = files[:max_files]
    return files

# (opcional) contar archivos:
# for name, (path, did) in DATASET_ROOTS.items():
#     print(name, len(scan_dataset_files(path)) if os.path.isdir(path) else 'missing')


## 3. Arquitecturas Corregidas (Mapeadas a los Checkpoints Reales)

* **Router**: Incluye `self.vit.norm` crítico y remueve devolución de `mask`.
* **Expertos**: Mapeos directos de bloque (`layer1`, etc.) para gradient checkpointing.

In [ ]:
"""
real_models.py — Arquitecturas reales del sistema MoE.

Extraídas de 05_Router_Profesor_Fase3_Solo.ipynb.
Incluye:
  - VisionRouter (SwitchablePatchEmbed + ViT-Tiny + Linear head)
  - LungMaxViT (Experto 1 — NIH)
  - build_efficientnet_b3_expert() (Experto 2 — ISIC)
  - build_vgg16_bn_expert() (Experto 3 — Osteoarthritis)
  - DCSwinBStyle3D (Experto 4 — LUNA16)
  - R3D18Expert (Experto 5 — Pancreas)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision.models as models
from torchvision.models.video import r3d_18
from torch.utils.checkpoint import checkpoint as grad_checkpoint

# Importar PatchEmbed de MONAI (necesario para SwitchablePatchEmbed)
try:
    from monai.networks.blocks import PatchEmbed
    HAS_MONAI = True
except ImportError:
    HAS_MONAI = False


# =============================================================================
# 1. Componentes del Router (VisionRouter)
# =============================================================================

class SwitchablePatchEmbed(nn.Module):
    """
    Switchable Patch Embedding para manejar entradas 2D y 3D.
    Usa MONAI PatchEmbed internamente.
    """

    def __init__(self, embed_dim=192, patch_size_2d=16, patch_size_3d=8,
                 in_channels_2d=3):
        super().__init__()
        if not HAS_MONAI:
            raise ImportError(
                "MONAI es necesario para SwitchablePatchEmbed. "
                "Instalar con: pip install monai"
            )
        self.embed_dim = embed_dim
        self.patch_embed_2d = PatchEmbed(
            spatial_dims=2,
            in_chans=in_channels_2d,
            patch_size=patch_size_2d,
            embed_dim=embed_dim,
        )
        self.patch_embed_3d = PatchEmbed(
            spatial_dims=3,
            in_chans=1,
            patch_size=patch_size_3d,
            embed_dim=embed_dim,
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 513, embed_dim))

    def _patch_tokens_to_sequence(self, patch_tokens: torch.Tensor) -> torch.Tensor:
        """Convierte la salida de PatchEmbed a una secuencia [T, embed_dim]."""
        x = patch_tokens
        if x.dim() == 3 and x.size(-1) == self.embed_dim:
            return x.squeeze(0) if x.size(0) == 1 else x.reshape(
                -1, self.embed_dim)
        if x.dim() >= 3 and x.size(-1) == self.embed_dim:
            return x.reshape(-1, self.embed_dim)
        if x.dim() == 4 and x.size(1) == self.embed_dim:
            x = x.flatten(2).transpose(1, 2).contiguous()
            return x.reshape(-1, self.embed_dim)
        if x.dim() == 5 and x.size(1) == self.embed_dim:
            x = x.flatten(2).transpose(1, 2).contiguous()
            return x.reshape(-1, self.embed_dim)
        return x

    def forward(self, batch_tensors):
        """
        batch_tensors: un tensor o lista de tensores.
        En el dashboard procesamos 1 sample a la vez.
        """
        if not isinstance(batch_tensors, (list, tuple)):
            batch_tensors = [batch_tensors]

        batch_size = len(batch_tensors)
        tokens_list = []

        for sample in batch_tensors:
            if sample.ndim == 3:  # [C, H, W] -> [1, C, H, W]
                sample = sample.unsqueeze(0)

            # Corregir casos donde un volumen 3D llega como 4D [1, D, H, W]
            # (El router espera 5D para patch_embed_3d)
            if sample.ndim == 4 and sample.shape[1] > 3 and sample.shape[1] == sample.shape[2] == sample.shape[3]:
                sample = sample.unsqueeze(1)  # [1, 64, 64, 64] -> [1, 1, 64, 64, 64]

            if sample.ndim == 4:  # 2D image
                if sample.shape[1] == 1:
                    sample = sample.repeat(1, 3, 1, 1)
                patch_tokens = self.patch_embed_2d(sample)
            elif sample.ndim == 5:  # 3D volume
                patch_tokens = self.patch_embed_3d(sample)
            else:
                raise ValueError(f"Dimensión de entrada inválida: {sample.ndim}")

            seq = self._patch_tokens_to_sequence(patch_tokens)
            tokens_list.append(seq)

        padded = torch.nn.utils.rnn.pad_sequence(
            tokens_list, batch_first=True)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, padded), dim=1)
        x = x + self.pos_embed[:, :x.size(1), :]
        return x


class VisionRouter(nn.Module):
    """
    VisionRouter basado en ViT-Tiny (vit_tiny_patch16_224 de timm).
    Incluye un hook para capturar los attention weights del último bloque
    (necesario para generar el Attention Heatmap, consigna #18).
    """

    def __init__(self, embed_dim=192, num_experts=5, pretrained=False,
                 **kwargs):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_experts = num_experts
        self.patch_embed = SwitchablePatchEmbed(embed_dim=embed_dim)

        self.vit = timm.create_model(
            'vit_tiny_patch16_224',
            pretrained=pretrained,
            num_classes=0,
            global_pool='',
        )
        # Reemplazar el patch embedding del ViT por Identity
        # porque usamos SwitchablePatchEmbed
        self.vit.patch_embed = nn.Identity()

        self.router_head = nn.Linear(embed_dim, num_experts)

        # Almacena el mapa de atención del último bloque
        self._last_attn_weights = None
        self._hook_handle = None
        self._register_attn_hook()

    def _register_attn_hook(self):
        """Registra un forward hook en el módulo de atención del último
        bloque del ViT para capturar los attention weights."""
        try:
            last_block = self.vit.blocks[-1]
            attn_module = last_block.attn

            def hook_fn(module, input, output):
                # En timm, el módulo Attention guarda attn_weights
                # internamente si configuramos attn_drop correctamente.
                # Usamos un enfoque alternativo: recalcular la atención.
                pass

            # Inyectamos un forward hook que captura qkv
            self._hook_handle = attn_module.register_forward_hook(
                self._attn_hook
            )
        except (AttributeError, IndexError):
            pass  # Si la estructura del ViT no lo permite, pasamos

    def _attn_hook(self, module, input, output):
        """Hook que captura los attention weights del módulo de atención."""
        # En timm ViT, la atención se computa internamente.
        # Necesitamos recalcularla a partir de qkv.
        try:
            x = input[0]
            B, N, C = x.shape
            qkv = module.qkv(x).reshape(
                B, N, 3, module.num_heads, C // module.num_heads
            ).permute(2, 0, 3, 1, 4)
            q, k, _ = qkv.unbind(0)
            head_dim = C // module.num_heads
            attn = (q @ k.transpose(-2, -1)) * (head_dim ** -0.5)
            attn = attn.softmax(dim=-1)
            self._last_attn_weights = attn.detach()
        except Exception:
            self._last_attn_weights = None

    def forward(self, batch_tensors):
        """
        Args:
            batch_tensors: tensor o lista de tensores preprocesados

        Returns:
            logits:       [B, num_experts] — gating logits
            cls_token:    [B, embed_dim] — CLS token embedding
            attn_weights: [B, num_heads, seq_len, seq_len] o None
        """
        x = self.patch_embed(batch_tensors)

        # Pasar por los bloques del ViT
        # (el hook capturará la atención del último bloque)
        for blk in self.vit.blocks:
            x = blk(x)

        # Eliminamos vit.norm para coincidir con la lógica del Notebook 05 §8.
        # Esto evitará el colapso del routing hacia el experto de ISIC.
        x = self.vit.norm(x)

        cls_token = x[:, 0]
        logits = self.router_head(cls_token)

        return logits, cls_token, self._last_attn_weights


# =============================================================================
# 2. Expertos Reales
# =============================================================================

class SwinNIHClassifier(nn.Module):
    """Experto 1 — NIH ChestX-ray. Swin-Tiny (notebook 05)."""

    def __init__(self, num_classes=5, pretrained=False):
        super().__init__()
        self.model = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=pretrained,
            num_classes=num_classes,
        )

    def forward(self, x):
        return self.model(x)


def build_efficientnet_b3_expert(num_classes=9):
    """Experto 2 — ISIC 2019. EfficientNet-B3."""
    model = timm.create_model(
        "efficientnet_b3",
        pretrained=False,
        num_classes=num_classes,
    )
    return model


def build_vgg16_bn_expert(num_classes=5):
    """
    Experto 3 — Osteoarthritis. VGG-16 BN con:
    - Entrada de 1 canal (radiografía)
    - Clasificador custom con BatchNorm y Dropout
    """
    model = models.vgg16_bn(weights=None)
    # Adaptar primera conv para 1 canal (igual que notebook 05)
    old_conv = model.features[0]
    new_conv = nn.Conv2d(1, 64, kernel_size=3, padding=1)
    with torch.no_grad():
        new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        new_conv.bias.copy_(old_conv.bias)
    model.features[0] = new_conv
    # Clasificador custom (del notebook)
    model.classifier = nn.Sequential(
        nn.Linear(512 * 7 * 7, 512),
        nn.ReLU(True),
        nn.BatchNorm1d(512),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(True),
        nn.BatchNorm1d(256),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )
    return model


class R3D18LunaKineticsExpert(nn.Module):
    """
    Experto 4 — LUNA16. R3D-18 binario: 3 canales con stats Kinetics.
    """

    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = r3d_18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        feat = self.backbone(x).flatten(1)
        return self.head(feat)


class R3D18Expert(nn.Module):
    """
    Experto 5 — Pancreas. R3D-18 con 3 canales de entrada.
    Usa gradient checkpointing obligatorio (consigna).
    
    El checkpoint usa keys directas (stem, layer1..4, avgpool)
    y un head secuencial con 128 unidades ocultas.
    """

    def __init__(self, num_classes=2):
        super().__init__()
        base = r3d_18(weights=None)
        
        # así que NO modificamos stem[0].
        
        # Atributos individuales para coincidir con keys del state_dict
        self.stem = base.stem
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.avgpool = base.avgpool
        
        # Head reconstruida: el checkpoint usa 128 unidades ocultas
        self.head = nn.Sequential(
            nn.Flatten(1),                  # 0
            nn.Linear(512, 128),           # 1 (weight, bias)
            nn.BatchNorm1d(128),           # 2 (weight, bias, running_mean...)
            nn.ReLU(True),                 # 3
            nn.Dropout(0.4),               # 4
            nn.Linear(128, num_classes)    # 5 (weight, bias)
        )

    def forward(self, x):
        if x.ndim == 4:
            x = x.unsqueeze(0)

        # Gradient checkpointing obligatorio por bloque (consigna §8.1)
        x = grad_checkpoint(self.stem, x, use_reentrant=False)
        x = grad_checkpoint(self.layer1, x, use_reentrant=False)
        x = grad_checkpoint(self.layer2, x, use_reentrant=False)
        x = grad_checkpoint(self.layer3, x, use_reentrant=False)
        x = grad_checkpoint(self.layer4, x, use_reentrant=False)

        x = self.avgpool(x)
        x = self.head(x)
        
        return x


import os
import glob
import subprocess
from typing import Dict, List, Tuple

EXPERT_SPECS = {
    1: {"name": "exp1_nih", "num_classes": 5, "arch": "swin_tiny_nih"},
    2: {"name": "exp2_isic", "num_classes": 9, "arch": "efficientnet_b3"},
    3: {"name": "exp3_osteo", "num_classes": 5, "arch": "vgg16_bn"},
    4: {
        "name": "exp4_luna16",
        "model_name": "R3D-18 (Kinetics 3ch)",
        "num_classes": 2,
        "arch": "r3d18_luna",
    },
    5: {"name": "exp5_pancreas", "num_classes": 2, "arch": "r3d18"},
}


def build_expert(expert_id: int, pretrained_backbone: bool = True):
    spec = EXPERT_SPECS[int(expert_id)]
    arch = spec["arch"]
    num_classes = spec["num_classes"]

    if arch == "swin_tiny_nih":
        return SwinNIHClassifier(num_classes=num_classes, pretrained=pretrained_backbone)
    if arch == "efficientnet_b3":
        return build_efficientnet_b3_expert(num_classes=num_classes)
    if arch == "vgg16_bn":
        return build_vgg16_bn_expert(num_classes=num_classes)
    if arch == "r3d18_luna":
        return R3D18LunaKineticsExpert(num_classes=num_classes, pretrained=pretrained_backbone)
    if arch == "r3d18":
        return R3D18Expert(num_classes=num_classes)
    raise ValueError(f"Arquitectura no soportada: {arch}")


def _default_checkpoint_candidates(weights_dir: str) -> Dict[int, List[str]]:
    """
    Candidatos por experto para tolerar nombres antiguos/nuevos de checkpoints.
    """
    return {
        1: [
            os.path.join(weights_dir, "Experts_2D", "MaxViT_NIH_5cls.pth"),
            os.path.join(weights_dir, "MaxViT_NIH_5cls.pth"),
            os.path.join(weights_dir, "exp1_NIH_SwinTiny_best.pth"),
            os.path.join(weights_dir, "exp1_NIH_LungMaxViT_best.pth"),
        ],
        2: [
            os.path.join(weights_dir, "exp2_ISIC_EfficientNetB3_best.pth"),
        ],
        3: [
            os.path.join(weights_dir, "exp3_Osteo_VGG16BN_best.pth"),
        ],
        4: [
            # Mismas rutas que antes; pesos del entrenamiento LUNA R3D-18 (expert4_*_best.pth)
            os.path.join(weights_dir, "exp4_LUNA16_3D_best.pth"),
        ],
        5: [
            os.path.join(weights_dir, "exp5_Pancreas_3D_best.pth"),
            os.path.join(weights_dir, "r3d18_pancreas_best_V2.pth"),
        ],
    }


def resolve_checkpoint(candidates: List[str]) -> str:
    """
    Busca el checkpoint. Si existe ``<ruta>.pth.zip`` y no el ``.pth``,
    descomprime en el mismo directorio y devuelve la ruta del ``.pth``.
    """
    for p in candidates:
        if any(ch in p for ch in ["*", "?", "["]):
            matches = sorted(glob.glob(p))
            if matches:
                return matches[0]
        if os.path.exists(p):
            return p
        zip_path = p + ".zip"
        if os.path.exists(zip_path):
            print(f"📦 Detectado peso comprimido: {os.path.basename(zip_path)}")
            extract_dir = os.path.dirname(zip_path) or "."
            try:
                print(f"  → Descomprimiendo en {extract_dir}...")
                subprocess.run(["unzip", "-o", zip_path, "-d", extract_dir], check=True)
                if os.path.exists(p):
                    print(f"  ✅ Checkpoint extraído: {os.path.basename(p)}")
                    return p
                new_matches = glob.glob(os.path.join(extract_dir, "*.pth"))
                if new_matches:
                    return sorted(new_matches)[0]
            except Exception as e:
                print(f"  ❌ Error al descomprimir {zip_path}: {e}")
    return ""


def load_weights(model: nn.Module, ckpt_path: str, map_location: str = "cpu", strict: bool = False):
    if not ckpt_path or not os.path.exists(ckpt_path):
        return False, "checkpoint no encontrado"
    raw = torch.load(ckpt_path, map_location=map_location)

    # Checkpoints de entrenamiento suelen ser dict con 'state_dict'
    if isinstance(raw, dict):
        if "state_dict" in raw:
            state = raw["state_dict"]
        elif "model_state_dict" in raw:
            state = raw["model_state_dict"]
        elif "model" in raw and isinstance(raw["model"], dict):
            state = raw["model"]
        else:
            state = raw
    else:
        state = raw

    try:
        model.load_state_dict(state, strict=strict)
        return True, "ok"
    except Exception as e:
        return False, f"load_state_dict fallo: {e}"

def freeze_and_eval(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    model.eval()
    return model


def load_all_experts_from_drive(
    weights_dir: str,
    device: str = "cpu",
    strict: bool = False,
    pretrained_backbone: bool = False,
) -> Tuple[Dict[int, nn.Module], Dict[int, dict]]:
    """
    Crea y carga los 5 expertos con pesos desde Drive.

    Retorna:
      experts: dict[int, nn.Module]
      info: dict[int, {arch, ckpt, loaded, params}]
    """
    candidates = _default_checkpoint_candidates(weights_dir)
    experts = {}
    info = {}

    for eid in sorted(EXPERT_SPECS.keys()):
        model = build_expert(eid, pretrained_backbone=pretrained_backbone)
        ckpt_path = resolve_checkpoint(candidates[eid])
        loaded, msg = load_weights(model, ckpt_path, map_location="cpu", strict=strict)

        model = freeze_and_eval(model).to(device)
        experts[eid] = model
        spec = EXPERT_SPECS[eid]
        info[eid] = {
            "name": spec["name"],
            "arch": spec["arch"],
            "model_name": spec.get("model_name", ""),
            "ckpt": ckpt_path,
            "loaded": loaded,
            "message": msg,
            "params": int(sum(p.numel() for p in model.parameters())),
        }

    return experts, info


def print_expert_load_report(info: Dict[int, dict]):
    print("=== Expert Load Report ===")
    for eid in sorted(info.keys()):
        row = info[eid]
        status = "OK" if row["loaded"] else "MISSING"
        mn = row.get("model_name") or ""
        mn_s = f" | {mn}" if mn else ""
        print(
            f"Exp{eid} | {row['name']} | arch={row['arch']}{mn_s} | {status} | "
            f"params={row['params']:,} | ckpt={row['ckpt'] or 'N/A'}"
        )
print("Arquitecturas de expertos: definidas en el notebook.")

def _default_checkpoint_candidates(weights_dir: str) -> Dict[int, List[str]]:
    return {
        1: [os.path.join(weights_dir, "exp1_NIH_Swin_Tiny_best.pth")],
        2: [os.path.join(weights_dir, "exp2_ISIC_EfficientNetB3_best.pth")],
        3: [os.path.join(weights_dir, "exp3_Osteo_VGG16BN_best.pth")],
        4: [os.path.join(weights_dir, "exp4_LUNA16_3D_best.pth")],
        5: [os.path.join(weights_dir, "exp5_Pancreas_3D_best.pth")]
    }

def resolve_checkpoint(candidates: List[str]) -> str:
    for p in candidates:
        if os.path.exists(p): return p
        zip_path = p + ".zip"
        if os.path.exists(zip_path):
            print(f"📦 Detectado peso comprimido: {os.path.basename(zip_path)}")
            extract_dir = os.path.dirname(zip_path) or "."
            try:
                subprocess.run(["unzip", "-o", zip_path, "-d", extract_dir], check=True, capture_output=True)
                if os.path.exists(p): return p
                # Si se extrae en una subcarpeta
                folder_path = p.replace(".pth", "")
                if os.path.exists(folder_path) and os.path.isdir(folder_path):
                    matches = glob.glob(os.path.join(folder_path, "*.pth"))
                    if matches: return sorted(matches)[0]
            except Exception as e:
                print(f"Error unzip: {e}")
    return ""


## 4. Administrador del Sistema MoE (Inferencia y Batches)

In [ ]:

import time

EXPERT_CLASSES = {
    1: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion'], # NIH (experto 1 en 05)
    2: ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC'], # ISIC (experto 2)
    3: ['KL0', 'KL1', 'KL2', 'KL3', 'KL4'], # Osteo (experto 3)
    4: ['Non-Nodule', 'Nodule'], # LUNA16 (experto 4)
    5: ['Normal', 'Tumor'] # Pancreas (experto 5)
}

class MoEProductionEngine:
    def __init__(self, repo_id="Slucu-0310/Proyecto-MoE-Pesos", weights_dir=WEIGHTS_DIR, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.repo_id = repo_id
        self.weights_dir = weights_dir
        self.device = device
        self.preprocessor = preprocessor
        self._download_from_hf()
        self._load_models()

    def _download_from_hf(self):
        print(f"📥 Sincronizando pesos desde Hugging Face (repo: {self.repo_id})...")
        from huggingface_hub import hf_hub_download
        os.makedirs(self.weights_dir, exist_ok=True)
        hf_files = [
            "router_professor_fase3_only_finetuned.pth",
            "exp1_NIH_Swin_Tiny_best.pth",
            "exp2_ISIC_EfficientNetB3_best.pth.zip",
            "exp3_Osteo_VGG16BN_best.pth",
            "exp4_LUNA16_3D_best.pth",
            "exp5_Pancreas_3D_best.pth"
        ]
        for f in hf_files:
            try:
                hf_hub_download(repo_id=self.repo_id, filename=f, local_dir=self.weights_dir)
            except Exception as e:
                print(f"⚠️ Advertencia: No se pudo descargar {f} desde HF. ({e})")

    def _load_models(self):
        print(f"📥 Cargando modelos en memoria desde {self.weights_dir}...")
        
        # 1. Router
        self.router = VisionRouter(num_experts=5, pretrained=False)
        router_ckpt_candidates = [
            os.path.join(self.weights_dir, "router_professor_fase3_only_finetuned.pth"),
            os.path.join(self.weights_dir, "router_professor.pth"),
            os.path.join(self.weights_dir, "router_best.pth")
        ]
        router_ckpt = next((p for p in router_ckpt_candidates if os.path.exists(p)), None)
        if router_ckpt is None:
            print(f"⚠️ ADVERTENCIA: No se encontró checkpoint del router en {self.weights_dir}.")
        else:
            load_weights(self.router, router_ckpt, strict=False)
        self.router = freeze_and_eval(self.router).to(self.device)
        
        # 2. Expertos
        print("📥 Cargando expertos...")
        self.experts, info = load_all_experts_from_drive(self.weights_dir, device=self.device, strict=False, pretrained_backbone=False)
        print_expert_load_report(info)
        print(f"✅ Todos los modelos cargados en {self.device}")

    def _prepare_expert_tensor(self, t_raw, expert_id, fpath):
        t = t_raw.to(self.device)
        if expert_id == 1:
            return t.unsqueeze(0)
        elif expert_id == 2:
            return t.unsqueeze(0)
        elif expert_id == 3:
            # Osteo VGG16_BN expects 1 channel. t_raw is [3, 224, 224]
            return t[0:1, :, :].unsqueeze(0)
        elif expert_id == 4:
            t = t.unsqueeze(0)
            t = luna_1ch_to_kinetics_3ch(t, device=self.device)
            return t
        elif expert_id == 5:
            # Pancreas R3D18 expects 3 channels. t_raw is [1, D, H, W]
            t = t.repeat(3, 1, 1, 1)
            return t.unsqueeze(0)
        return t.unsqueeze(0)

    @torch.no_grad()
    def predict_batch(self, file_paths):
        """Realiza predicciones sobre una lista de archivos (Batch Inference)."""
        results = []
        tensors_raw = []
        valid_paths = []
        
        print(f"🔄 Preprocesando {len(file_paths)} archivos...")
        for path in file_paths:
            try:
                t = self.preprocessor(path)
                tensors_raw.append(t)
                valid_paths.append(path)
            except Exception as e:
                print(f"❌ Error procesando {path}: {e}")
                
        if not tensors_raw: return []

        print("🧠 Enrutando batch...")
        # Agregar dimensión de batch manualmente para que el SwitchablePatchEmbed reconozca las 3D
        router_tensors = [t.unsqueeze(0).to(self.device) for t in tensors_raw]
        # El router devuelve logits, cls_token, attn_weights
        logits, _, _ = self.router(router_tensors)
        
        gating_probs = F.softmax(logits, dim=-1)
        expert_indices = gating_probs.argmax(dim=1).cpu().numpy()
        
        print("🔬 Evaluando en expertos...")
        for i, path in enumerate(valid_paths):
            idx = int(expert_indices[i])
            eid = idx + 1
            
            t_exp = self._prepare_expert_tensor(tensors_raw[i], eid, path)
            expert = self.experts.get(eid)
            if expert is None:
                continue
                
            expert_logits = expert(t_exp)
            class_probs = F.softmax(expert_logits, dim=-1).squeeze(0).cpu().numpy()
            
            class_idx = int(class_probs.argmax())
            confidence = float(class_probs.max())
            class_name = EXPERT_CLASSES[eid][class_idx]
            
            results.append({
                "file": os.path.basename(path),
                "expert_id": eid,
                "class_idx": class_idx,
                "class_name": class_name,
                "confidence": confidence,
                "gating_probs": gating_probs[i].cpu().numpy()
            })
            
        return results


## 5. Prueba de Ejecución (Batch Inference)

In [ ]:

moe = MoEProductionEngine()
print("Clase MoEProductionEngine inicializada correctamente.")
